# How LLMs Work — Hands-On Practice

This notebook lets you **touch and feel** how a real language model works — from tokenization to text generation to attention.

**Model used:** GPT-2 Small (124M parameters) — fast enough on a free Colab T4 GPU.

**What we'll cover:**
1. Setup & model loading
2. Tokenization — how text becomes numbers
3. Next-token prediction — what the model actually outputs
4. Decoding strategies — greedy vs. sampling vs. beam search
5. Perplexity — measuring how "surprised" the model is
6. Attention visualization — what the model "looks at"
7. Prompt engineering — zero-shot and few-shot
8. Embeddings — words as vectors in space

---
## Part 0: Setup

Install the HuggingFace libraries we need. This takes ~30 seconds on Colab.

In [ ]:
# Run this cell first — installs required packages
!pip install -q transformers accelerate bertviz

import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

---
## Part 1: Load GPT-2

GPT-2 Small has **124M parameters** and fits comfortably on a free Colab GPU (or even CPU).
We load both the **model** (neural network weights) and the **tokenizer** (text ↔ token converter).

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

MODEL_NAME = 'gpt2'   # 124M params — swap 'gpt2-medium' (345M) if you have more VRAM

print('Loading tokenizer...')
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 has no pad token by default

print('Loading model...')
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME, output_attentions=True)
model = model.to(device)
model.eval()   # inference mode — no dropout

n_params = sum(p.numel() for p in model.parameters())
print(f'\nModel loaded! Parameters: {n_params / 1e6:.1f}M')
print(f'Architecture: {model.config.n_layer} layers, '
      f'{model.config.n_head} attention heads, '
      f'd_model={model.config.n_embd}')

---
## Part 2: Tokenization — Text to Numbers

LLMs don't read characters or words — they read **tokens**.
GPT-2 uses **Byte Pair Encoding (BPE)**: common words become one token, rare words split into subwords.

The vocabulary size of GPT-2 is **50,257 tokens**.

In [ ]:
sentences = [
    'Hello, world!',
    'The quick brown fox jumps over the lazy dog.',
    'Transformer architecture revolutionized NLP.',
    'Unbelievably, supercalifragilistic is one word.',
]

print(f'Vocabulary size: {tokenizer.vocab_size:,}\n')
print('=' * 60)

for sent in sentences:
    token_ids = tokenizer.encode(sent)
    tokens    = tokenizer.convert_ids_to_tokens(token_ids)
    print(f'Text   : {sent}')
    print(f'Tokens : {tokens}')
    print(f'IDs    : {token_ids}')
    print(f'Count  : {len(token_ids)} tokens')
    print('-' * 60)

In [ ]:
# Visualize token boundaries by color
def visualize_tokens(text, tokenizer):
    token_ids = tokenizer.encode(text)
    tokens    = [tokenizer.decode([tid]) for tid in token_ids]

    colors = plt.cm.Set3(np.linspace(0, 1, len(tokens)))
    fig, ax = plt.subplots(figsize=(max(10, len(tokens) * 0.8), 1.2))
    ax.set_xlim(0, len(tokens))
    ax.set_ylim(0, 1)
    ax.axis('off')

    for i, (tok, col) in enumerate(zip(tokens, colors)):
        rect = plt.Rectangle((i, 0.2), 0.95, 0.6,
                               facecolor=col, edgecolor='gray', linewidth=0.8)
        ax.add_patch(rect)
        display_text = tok.replace('Ġ', '_').replace('Ċ', '\\n')
        ax.text(i + 0.475, 0.5, display_text, ha='center', va='center',
                fontsize=9, fontweight='bold')
        ax.text(i + 0.475, 0.1, str(token_ids[i]), ha='center', va='center',
                fontsize=7, color='gray')

    ax.set_title(f'Tokenization of: "{text}"  ({len(tokens)} tokens)', fontsize=11)
    plt.tight_layout()
    plt.show()

visualize_tokens('Large Language Models are powerful.', tokenizer)
visualize_tokens('Unbelievably supercalifragilistic!', tokenizer)

---
## Part 3: Next-Token Prediction

At its core, GPT-2 does ONE thing: given a sequence of tokens, predict the **probability distribution over the next token**.

The output is a vector of size **50,257** (the vocab size) — a score for every possible next token.
We apply **softmax** to turn scores into probabilities.

In [ ]:
def get_top_next_tokens(prompt, tokenizer, model, top_k=10):
    """Run one forward pass and return the top-k predicted next tokens."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    # logits shape: (batch=1, seq_len, vocab_size)
    # We want the logits for the LAST token position
    last_logits = outputs.logits[0, -1, :]          # shape: (vocab_size,)
    probs       = torch.softmax(last_logits, dim=-1) # convert to probabilities

    top_probs, top_ids = torch.topk(probs, top_k)
    top_tokens = [tokenizer.decode([tid.item()]) for tid in top_ids]
    return list(zip(top_tokens, top_probs.tolist()))


def plot_top_tokens(prompt, tokenizer, model, top_k=10):
    results = get_top_next_tokens(prompt, tokenizer, model, top_k)
    tokens, probs = zip(*results)
    tokens = [repr(t) for t in tokens]   # show whitespace clearly

    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(tokens[::-1], probs[::-1], color='steelblue')
    ax.set_xlabel('Probability', fontsize=11)
    ax.set_title(f'Top {top_k} next tokens for: "{prompt}"', fontsize=11)
    for bar, prob in zip(bars, probs[::-1]):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{prob:.3f}', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()


# Try different prompts — notice how the model's "certainty" changes
plot_top_tokens('The capital of France is', tokenizer, model)
plot_top_tokens('I went to the store to buy some', tokenizer, model)

In [ ]:
# --- Try your own prompt! ---
MY_PROMPT = 'The president of the United States is'   # <-- change this
plot_top_tokens(MY_PROMPT, tokenizer, model, top_k=12)

---
## Part 4: Decoding Strategies

Once we have the probability distribution over the next token, we need to **pick** a token.
The strategy we use dramatically changes the output:

| Strategy | How it picks | Effect |
|---|---|---|
| **Greedy** | Always pick the highest-probability token | Deterministic, often repetitive |
| **Sampling** | Sample randomly from the distribution | Creative, but can be incoherent |
| **Top-k sampling** | Sample from top-k tokens only | Balances quality and diversity |
| **Top-p (nucleus)** | Sample from smallest set covering ≥p probability | Adaptive, popular in practice |
| **Temperature** | Scale logits before softmax | Controls sharpness of distribution |
| **Beam search** | Keep top-b candidates at each step | Better quality, less diverse |

In [ ]:
from transformers import set_seed

PROMPT = 'Once upon a time in a land far away,'
MAX_NEW_TOKENS = 60

inputs = tokenizer(PROMPT, return_tensors='pt').to(device)

strategies = {
    'Greedy': dict(do_sample=False),
    'Sampling (temp=1.0)': dict(do_sample=True, temperature=1.0),
    'Sampling (temp=0.3)': dict(do_sample=True, temperature=0.3),
    'Top-k (k=50)': dict(do_sample=True, top_k=50),
    'Top-p / Nucleus (p=0.9)': dict(do_sample=True, top_p=0.9, top_k=0),
    'Beam search (beams=4)': dict(do_sample=False, num_beams=4),
}

set_seed(42)
print(f'Prompt: "{PROMPT}"\n')
print('=' * 70)

for name, kwargs in strategies.items():
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            pad_token_id=tokenizer.eos_token_id,
            **kwargs
        )
    # Only decode the newly generated tokens
    new_ids = output_ids[0][inputs['input_ids'].shape[1]:]
    generated = tokenizer.decode(new_ids, skip_special_tokens=True)
    print(f'[{name}]')
    print(generated)
    print('-' * 70)

In [ ]:
# Visualize how temperature reshapes the probability distribution
prompt = 'The weather today is'
inputs = tokenizer(prompt, return_tensors='pt').to(device)

with torch.no_grad():
    logits = model(**inputs).logits[0, -1, :].cpu()

temperatures = [0.3, 0.7, 1.0, 1.5]
top_k = 15

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharey=False)

# Get consistent top-k from T=1 for comparison
base_probs = torch.softmax(logits, dim=-1)
top_ids = torch.topk(base_probs, top_k).indices
top_tokens = [repr(tokenizer.decode([i.item()])) for i in top_ids]

for ax, T in zip(axes, temperatures):
    scaled_probs = torch.softmax(logits / T, dim=-1)
    probs_shown  = scaled_probs[top_ids].numpy()
    ax.barh(top_tokens[::-1], probs_shown[::-1], color='steelblue')
    ax.set_title(f'Temperature = {T}', fontsize=11)
    ax.set_xlabel('Probability')
    if T != 0.3:
        ax.set_yticklabels([])

fig.suptitle(f'Temperature Effect on Distribution — prompt: "{prompt}"', fontsize=12)
plt.tight_layout()
plt.show()

print('Low temperature → more concentrated (greedy-like)')
print('High temperature → more spread out (creative/random)')

### How does temperature actually work?

Before the softmax, the model outputs raw **logits** — unnormalized scores for every token.  
Temperature `T` is applied by **dividing** those logits before the softmax:

$$p_i = \frac{e^{z_i / T}}{\sum_j e^{z_j / T}}$$

| Temperature | Effect on logits | Effect on distribution |
|---|---|---|
| `T → 0` | Differences blown up to ±∞ | All mass on argmax → **identical to greedy** |
| `T = 1` | Logits unchanged | Original model distribution |
| `T > 1` | Differences shrink | Flatter — more tokens get real probability |
| `T → ∞` | All logits → 0 | Uniform over whole vocabulary |

**Key intuition:** A small T *amplifies* the gaps between logits, so the highest-scoring token  
dominates almost completely. Think of it like turning up the contrast on an image.

In [ ]:
# Step-by-step view: raw logits → scaled → softmax
# We use a toy 6-token example so the numbers are easy to read

import numpy as np
import matplotlib.pyplot as plt

# Realistic-looking logits for 6 candidate tokens
token_labels = ['Paris', 'London', 'Rome', 'Berlin', 'Tokyo', 'pizza']
raw_logits   = np.array([8.2, 6.5, 5.8, 5.1, 4.3, 1.9], dtype=float)

temperatures = [0.1, 0.5, 1.0, 2.0, 5.0]
colors = plt.cm.RdYlBu_r(np.linspace(0.1, 0.9, len(temperatures)))

def softmax(x):
    e = np.exp(x - x.max())   # subtract max for numerical stability
    return e / e.sum()


fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: Raw logits (fixed, T-independent) ──────────────────
ax = axes[0]
ax.bar(token_labels, raw_logits, color='slategray')
ax.set_title('Raw Logits (before temperature)', fontsize=11)
ax.set_ylabel('Logit value')
for i, v in enumerate(raw_logits):
    ax.text(i, v + 0.1, f'{v}', ha='center', fontsize=9)
ax.set_ylim(0, 11)
ax.grid(axis='y', linestyle='--', alpha=0.4)

# ── Panel 2: Scaled logits z/T ──────────────────────────────────
ax = axes[1]
x = np.arange(len(token_labels))
width = 0.15
for i, (T, col) in enumerate(zip(temperatures, colors)):
    scaled = raw_logits / T
    offset = (i - len(temperatures)//2) * width
    ax.bar(x + offset, scaled, width=width, color=col, label=f'T={T}')
ax.set_title('Scaled Logits  z / T', fontsize=11)
ax.set_ylabel('Scaled logit')
ax.set_xticks(x)
ax.set_xticklabels(token_labels)
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

# ── Panel 3: Resulting probabilities after softmax ───────────────
ax = axes[2]
for i, (T, col) in enumerate(zip(temperatures, colors)):
    probs  = softmax(raw_logits / T)
    offset = (i - len(temperatures)//2) * width
    ax.bar(x + offset, probs, width=width, color=col, label=f'T={T}')
ax.set_title('Probability after Softmax  exp(z/T) / Σ', fontsize=11)
ax.set_ylabel('Probability')
ax.set_xticks(x)
ax.set_xticklabels(token_labels)
ax.legend(fontsize=8, loc='upper right')
ax.grid(axis='y', linestyle='--', alpha=0.4)
ax.set_ylim(0, 1.05)
ax.axhline(1/len(token_labels), color='gray', linestyle=':', alpha=0.7, label='Uniform')

fig.suptitle(
    'Temperature Scaling: How T reshapes the logit → probability mapping\n'
    'Small T concentrates probability on the top token; large T spreads it out.',
    fontsize=12
)
plt.tight_layout()
plt.show()


# ── Print numbers explicitly ─────────────────────────────────────
print(f'Raw logits: {dict(zip(token_labels, raw_logits))}\n')
print(f'{"T":<6}  ' + '  '.join(f'{t:<8}' for t in token_labels))
print('-' * 70)
for T in temperatures:
    probs = softmax(raw_logits / T)
    row   = '  '.join(f'{p:<8.4f}' for p in probs)
    print(f'{T:<6}  {row}')

In [ ]:
# What happens as T → 0?  The gap between the top logit and the rest grows without bound.
# Visualize how P(top token) and entropy change with T

import scipy.stats

ts = np.logspace(-1.5, 1.0, 200)   # T from ~0.03 to 10

top_probs   = []
entropies   = []

for T in ts:
    probs = softmax(raw_logits / T)
    top_probs.append(probs[0])                       # probability of 'Paris'
    entropies.append(scipy.stats.entropy(probs, base=2))  # bits

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Annotate key temperatures
key_temps = {0.1: 'T=0.1\n(near-greedy)', 0.5: 'T=0.5', 1.0: 'T=1\n(original)', 2.0: 'T=2', 5.0: 'T=5'}

ax1.plot(ts, top_probs, color='steelblue', linewidth=2)
ax1.set_xscale('log')
ax1.set_xlabel('Temperature T (log scale)', fontsize=11)
ax1.set_ylabel('P("Paris") — top token probability', fontsize=11)
ax1.set_title('As T → 0, the top token gets nearly all the probability', fontsize=11)
ax1.grid(True, which='both', linestyle='--', alpha=0.4)
ax1.axhline(1.0, color='tomato', linestyle='--', alpha=0.5, label='P=1 (greedy limit)')
for T, label in key_temps.items():
    p = softmax(raw_logits / T)[0]
    ax1.annotate(label, xy=(T, p), xytext=(T*1.3, p - 0.08),
                 fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))
ax1.legend(fontsize=9)

ax2.plot(ts, entropies, color='darkorange', linewidth=2)
ax2.set_xscale('log')
ax2.set_xlabel('Temperature T (log scale)', fontsize=11)
ax2.set_ylabel('Entropy (bits)', fontsize=11)
ax2.set_title('Entropy: how much randomness / diversity in the sampling', fontsize=11)
ax2.grid(True, which='both', linestyle='--', alpha=0.4)
ax2.axhline(np.log2(len(raw_logits)), color='tomato', linestyle='--',
             alpha=0.5, label=f'Max entropy = {np.log2(len(raw_logits)):.1f} bits (uniform)')
ax2.axhline(0, color='navy', linestyle='--', alpha=0.4, label='0 bits (greedy limit)')
ax2.legend(fontsize=9)

plt.suptitle('Effect of Temperature on Top-Token Probability and Entropy', fontsize=12)
plt.tight_layout()
plt.show()

print('Takeaway:')
print('  T < 1  → top-token probability rises sharply; less diversity (low entropy)')
print('  T = 1  → original model distribution')
print('  T > 1  → probabilities flatten; more diversity, more risk of incoherence')
print('  T → 0  → equivalent to greedy decoding (argmax)')

---
## Part 5: Perplexity — How Surprised Is the Model?

**Perplexity (PPL)** measures how well a language model predicts a text.

$$\text{PPL}(x) = \exp\left(-\frac{1}{N} \sum_{i=1}^{N} \log P(x_i \mid x_{<i})\right)$$

- Low PPL → model found the text very predictable (e.g., common English)
- High PPL → model was surprised (e.g., random text, different language)

GPT-2 was trained on English web text (WebText). It will have **low perplexity on natural English** and **high perplexity on unusual text**.

In [ ]:
def compute_perplexity(text, tokenizer, model):
    """Compute perplexity of a text under the model."""
    inputs = tokenizer(text, return_tensors='pt').to(device)
    input_ids = inputs['input_ids']

    with torch.no_grad():
        outputs = model(input_ids, labels=input_ids)

    # outputs.loss = mean negative log-likelihood per token
    ppl = torch.exp(outputs.loss).item()
    return ppl


texts = {
    'Natural English sentence': 'The quick brown fox jumps over the lazy dog.',
    'News-style text': 'The president signed the bill into law on Tuesday afternoon.',
    'Grammatical but unlikely': 'The colorless green ideas sleep furiously today.',
    'Random words': 'Banana telescope purple umbrella seven fly mountain.',
    'Random characters': 'xkq zwp vbt mnr olk fjh cde.',
    'French text': 'Le chat est assis sur le tapis rouge.',
    'Code (Python)': 'def add(a, b): return a + b',
    'Repeated text': 'the the the the the the the the the.',
}

print(f'{"Text":<45} {"Perplexity":>12}')
print('-' * 60)
results = {}
for label, text in texts.items():
    ppl = compute_perplexity(text, tokenizer, model)
    results[label] = ppl
    print(f'{label:<45} {ppl:>12.1f}')

# Bar chart
fig, ax = plt.subplots(figsize=(10, 5))
labels, ppls = zip(*sorted(results.items(), key=lambda x: x[1]))
colors = ['steelblue' if p < 100 else ('darkorange' if p < 500 else 'tomato')
          for p in ppls]
ax.barh(labels, ppls, color=colors)
ax.set_xlabel('Perplexity (lower = model knows this kind of text better)', fontsize=11)
ax.set_title('GPT-2 Perplexity on Different Text Types', fontsize=12)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

---
## Part 6: Attention Visualization

The Transformer's core operation is **self-attention**: each token decides how much to "attend to" every other token.

GPT-2 has:
- **12 layers** of attention
- **12 attention heads** per layer

Each head learns to focus on different linguistic patterns (e.g., previous word, subject-verb agreement, coreference).

In [ ]:
def get_attention(text, tokenizer, model):
    """Return attention weights for all layers and heads."""
    inputs = tokenizer(text, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    # Replace GPT-2's Ġ (space prefix) for readability
    tokens = [t.replace('Ġ', ' ') for t in tokens]

    with torch.no_grad():
        outputs = model(**inputs, output_attentions=True)

    # attentions: tuple of (1, n_heads, seq_len, seq_len) per layer
    attentions = torch.stack([a.squeeze(0).cpu() for a in outputs.attentions])
    # shape: (n_layers, n_heads, seq_len, seq_len)
    return attentions, tokens


def plot_attention_heatmap(text, layer, head, tokenizer, model):
    attentions, tokens = get_attention(text, tokenizer, model)
    attn_matrix = attentions[layer, head].numpy()   # (seq_len, seq_len)

    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(attn_matrix, cmap='Blues', vmin=0, vmax=attn_matrix.max())
    ax.set_xticks(range(len(tokens)))
    ax.set_yticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticklabels(tokens, fontsize=9)
    ax.set_xlabel('Keys (attended TO)', fontsize=10)
    ax.set_ylabel('Queries (attending FROM)', fontsize=10)
    ax.set_title(f'Attention Weights — Layer {layer}, Head {head}\n"{text}"', fontsize=11)
    plt.colorbar(im, ax=ax, fraction=0.046)
    plt.tight_layout()
    plt.show()


TEXT = 'The cat sat on the mat because it was tired.'

# Show a few different layers/heads — each captures different patterns
plot_attention_heatmap(TEXT, layer=0,  head=0,  tokenizer=tokenizer, model=model)
plot_attention_heatmap(TEXT, layer=6,  head=3,  tokenizer=tokenizer, model=model)
plot_attention_heatmap(TEXT, layer=11, head=11, tokenizer=tokenizer, model=model)

In [ ]:
# Compare all 12 heads in a single layer
def plot_all_heads(text, layer, tokenizer, model):
    attentions, tokens = get_attention(text, tokenizer, model)
    n_heads = attentions.shape[1]

    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    for head, ax in enumerate(axes.flat):
        attn = attentions[layer, head].numpy()
        ax.imshow(attn, cmap='Blues', vmin=0)
        ax.set_xticks(range(len(tokens)))
        ax.set_yticks(range(len(tokens)))
        ax.set_xticklabels(tokens, rotation=90, fontsize=6)
        ax.set_yticklabels(tokens, fontsize=6)
        ax.set_title(f'Head {head}', fontsize=9)

    fig.suptitle(f'All 12 Attention Heads — Layer {layer}\n"{text}"', fontsize=12)
    plt.tight_layout()
    plt.show()

plot_all_heads('The dog chased the cat up the tree.', layer=5, tokenizer=tokenizer, model=model)

---
## Part 7: Prompt Engineering — Zero-Shot and Few-Shot

GPT-2 was not instruction-tuned (no RLHF), but we can still demonstrate zero-shot and few-shot prompting patterns.

The key insight: the model continues whatever pattern it sees in the prompt.

In [ ]:
def generate(prompt, tokenizer, model, max_new_tokens=80, temperature=0.7, top_p=0.9):
    """Generate text with top-p sampling."""
    inputs = tokenizer(prompt, return_tensors='pt').to(device)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            top_k=0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids   = output_ids[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


# ── Zero-shot ──────────────────────────────────────────────────
zero_shot_prompt = """Translate English to French:
English: Hello, how are you?
French:"""

print('[ZERO-SHOT]')
print('Prompt:', repr(zero_shot_prompt))
print('Generated:', generate(zero_shot_prompt, tokenizer, model, max_new_tokens=20))
print()

# ── Few-shot ───────────────────────────────────────────────────
few_shot_prompt = """Classify the sentiment:
Text: I love this movie! Sentiment: Positive
Text: This was terrible. Sentiment: Negative
Text: It was okay, not great. Sentiment: Neutral
Text: Absolutely fantastic experience! Sentiment:"""

print('[FEW-SHOT — Sentiment Classification]')
print('Prompt:')
print(few_shot_prompt)
print('Generated:', generate(few_shot_prompt, tokenizer, model, max_new_tokens=5))
print()

# ── Pattern continuation ───────────────────────────────────────
pattern_prompt = """Q: What is 2 + 2?
A: 4
Q: What is 3 + 5?
A: 8
Q: What is 10 + 7?
A:"""

print('[PATTERN CONTINUATION — Math]')
print(pattern_prompt)
print('Generated:', generate(pattern_prompt, tokenizer, model, max_new_tokens=2))
print()

pattern_prompt2 = """Q: What is 2 + 2?
A: 4
Q: What is 3 + 5?
A: 8
Q: What is 7 + 10?
A:"""

print('[PATTERN CONTINUATION — Math]')
print(pattern_prompt2)
print('Generated:', generate(pattern_prompt2, tokenizer, model, max_new_tokens=2))
print()

pattern_prompt_wo_examples = """Q: What is 10 + 7?
A:"""

print('[Zero-SHOT — Math]')
print(pattern_prompt_wo_examples)
print('Generated:', generate(pattern_prompt_wo_examples, tokenizer, model, max_new_tokens=10))
print()

---
## Part 8: Word Embeddings — Words as Vectors

The first thing GPT-2 does is look up each token in an **embedding table** — converting it to a dense vector of size 768.

Words with similar meanings end up near each other in this vector space.
We can see this by extracting embeddings and computing cosine similarity.

In [ ]:
import torch.nn.functional as F
from sklearn.decomposition import PCA

# Get the raw token embedding matrix (50257, 768)
embedding_matrix = model.transformer.wte.weight.data.cpu()  # (vocab_size, d_model)
print(f'Embedding matrix shape: {embedding_matrix.shape}')


def get_embedding(word, tokenizer, embedding_matrix):
    """Get the embedding for a single word (first token if multi-token)."""
    token_ids = tokenizer.encode(' ' + word)  # space prefix = word in mid-sentence
    return embedding_matrix[token_ids[0]]


def cosine_sim(a, b):
    return F.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()


# Semantic similarity pairs
pairs = [
    ('king',   'queen'),
    ('king',   'man'),
    ('king',   'car'),
    ('dog',    'cat'),
    ('dog',    'puppy'),
    ('dog',    'democracy'),
    ('happy',  'joyful'),
    ('happy',  'sad'),
    ('Paris',  'France'),
    ('Paris',  'Berlin'),
    ('Paris',  'banana'),
]

print(f'{"Word 1":<12} {"Word 2":<12} {"Cosine Similarity":>20}')
print('-' * 46)
for w1, w2 in pairs:
    e1 = get_embedding(w1, tokenizer, embedding_matrix)
    e2 = get_embedding(w2, tokenizer, embedding_matrix)
    sim = cosine_sim(e1, e2)
    print(f'{w1:<12} {w2:<12} {sim:>20.4f}')

In [ ]:
# Visualize embedding clusters with PCA
word_groups = {
    'Royalty':   ['king', 'queen', 'prince', 'princess', 'throne'],
    'Animals':   ['dog', 'cat', 'horse', 'fish', 'bird'],
    'Countries': ['France', 'Germany', 'Japan', 'Brazil', 'Canada'],
    'Emotions':  ['happy', 'sad', 'angry', 'fear', 'joy'],
    'Numbers':   ['one', 'two', 'three', 'four', 'five'],
}

all_words  = [w for group in word_groups.values() for w in group]
all_embeds = np.stack([get_embedding(w, tokenizer, embedding_matrix).numpy()
                        for w in all_words])

# Reduce to 2D with PCA
pca = PCA(n_components=2)
coords = pca.fit_transform(all_embeds)

colors = plt.cm.Set1(np.linspace(0, 1, len(word_groups)))
fig, ax = plt.subplots(figsize=(10, 8))

idx = 0
for (group_name, words), color in zip(word_groups.items(), colors):
    n = len(words)
    xs, ys = coords[idx:idx+n, 0], coords[idx:idx+n, 1]
    ax.scatter(xs, ys, color=color, s=100, label=group_name, zorder=5)
    for word, x, y in zip(words, xs, ys):
        ax.annotate(word, (x, y), textcoords='offset points',
                    xytext=(6, 4), fontsize=9, color=color)
    idx += n

ax.set_title('GPT-2 Token Embeddings (PCA to 2D)\nSimilar words cluster together', fontsize=12)
ax.legend(fontsize=10)
ax.grid(True, linestyle='--', alpha=0.4)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
plt.tight_layout()
plt.show()

In [ ]:
# The classic analogy: king - man + woman ≈ queen
# This works in Word2Vec; let's see how GPT-2 embeddings do

def analogy(a, b, c, tokenizer, embedding_matrix, top_n=5):
    """
    Find d such that: a - b + c ≈ d
    Example: king - man + woman ≈ queen
    """
    ea = get_embedding(a, tokenizer, embedding_matrix)
    eb = get_embedding(b, tokenizer, embedding_matrix)
    ec = get_embedding(c, tokenizer, embedding_matrix)
    target = ea - eb + ec

    # Cosine similarity to all embeddings
    norms  = embedding_matrix.norm(dim=1, keepdim=True).clamp(min=1e-8)
    normed = embedding_matrix / norms
    target_normed = target / target.norm().clamp(min=1e-8)
    sims = (normed @ target_normed).numpy()

    # Exclude the input words themselves
    exclude_ids = set(tokenizer.encode(f' {a}') +
                      tokenizer.encode(f' {b}') +
                      tokenizer.encode(f' {c}'))

    top_indices = np.argsort(sims)[::-1]
    results = []
    for idx in top_indices:
        if idx not in exclude_ids:
            word = tokenizer.decode([idx]).strip()
            if word.isalpha():   # skip tokens with punctuation/numbers
                results.append((word, sims[idx]))
        if len(results) >= top_n:
            break
    return results


analogies = [
    ('king',   'man',    'woman'),   # → queen?
    ('Paris',  'France', 'Germany'), # → Berlin?
    ('walked', 'walk',   'run'),     # → ran?
]

for a, b, c in analogies:
    results = analogy(a, b, c, tokenizer, embedding_matrix)
    print(f'{a} - {b} + {c} ≈ ', end='')
    print(', '.join(f'{w} ({s:.3f})' for w, s in results))

---
## Part 9: Contextual vs. Static Embeddings

Unlike Word2Vec (static), GPT-2 produces **contextual embeddings** — the same word gets a different vector depending on its context.

Example: the word **"bank"** means different things in different sentences.

In [ ]:
def get_contextual_embedding(sentence, target_word, tokenizer, model, layer=-1):
    """Return the hidden state of target_word in a sentence."""
    inputs = tokenizer(sentence, return_tensors='pt').to(device)
    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)

    # hidden_states: tuple of (1, seq_len, d_model) per layer, including embedding layer
    hidden = outputs.hidden_states[layer][0].cpu()  # (seq_len, d_model)

    # Find the token position of target_word
    for i, tok in enumerate(tokens):
        if target_word.lower() in tok.lower().replace('ġ', ''):
            return hidden[i]
    return hidden[0]  # fallback


contexts = {
    'river bank':     'I sat on the bank of the river.',
    'money bank':     'She withdrew cash from the bank.',
    'bank shot':      'He made a bank shot in billiards.',
    'data bank':      'The data bank stores millions of records.',
}

embeddings = {}
for label, sentence in contexts.items():
    emb = get_contextual_embedding(sentence, 'bank', tokenizer, model, layer=12)
    embeddings[label] = emb

# Pairwise cosine similarities
labels = list(embeddings.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

for i, l1 in enumerate(labels):
    for j, l2 in enumerate(labels):
        sim_matrix[i, j] = cosine_sim(embeddings[l1], embeddings[l2])

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap='RdYlGn', vmin=0.7, vmax=1.0)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=10)
ax.set_yticklabels(labels, fontsize=10)
for i in range(n):
    for j in range(n):
        ax.text(j, i, f'{sim_matrix[i, j]:.3f}', ha='center', va='center', fontsize=10)
plt.colorbar(im, ax=ax)
ax.set_title('Contextual Similarity of "bank" in different sentences\n(Layer 12 hidden states)', fontsize=11)
plt.tight_layout()
plt.show()

print('\nIf GPT-2 understands context, similar senses of "bank" should have higher cosine similarity.')

---
## Recap

| Concept | What we did | Key takeaway |
|---|---|---|
| **Tokenization** | Encoded text, visualized BPE splits | Models see subword tokens, not characters/words |
| **Next-token prediction** | Inspected logits + softmax | Every forward pass outputs a full vocab distribution |
| **Decoding strategies** | Compared greedy / sampling / beam | Strategy choice dramatically affects output quality |
| **Temperature** | Varied T, plotted distributions | Low T = safe/repetitive, High T = creative/risky |
| **Perplexity** | Scored diverse text types | Model assigns low PPL to text similar to its training data |
| **Attention** | Plotted head-level heatmaps | Different heads capture different linguistic patterns |
| **Prompt engineering** | Zero-shot / few-shot prompts | Pattern in context shapes what the model continues |
| **Embeddings** | Cosine sim, PCA, analogies | Semantic meaning encoded as geometry in vector space |
| **Contextual embeddings** | Same word, different sentences | Same token → different hidden vector depending on context |

### Next steps
- Try a larger model: `gpt2-medium` (345M) or `gpt2-large` (774M)
- Explore an instruction-tuned model: `microsoft/phi-2` or `TinyLlama/TinyLlama-1.1B-Chat-v1.0`
- Fine-tune GPT-2 on a custom dataset with HuggingFace `Trainer`
- Read the original papers: [Attention Is All You Need](https://arxiv.org/abs/1706.03762), [GPT-2](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf)

---
## Bonus: Load Any HuggingFace Model in 3 Lines

HuggingFace `pipeline` is the fastest way to load a model and start prompting — no tokenizer or model setup required.

```
pip install transformers accelerate
```

Browse models at [huggingface.co/models](https://huggingface.co/models) and swap in any `text-generation` model ID below.

In [ ]:
from transformers import pipeline

# ----- Choose your model -----
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # ~2 GB, fits on most laptops. Please change the model ID which
# Other lightweight options:
#   "microsoft/phi-2"             (~5 GB)
#   "google/gemma-2b-it"          (~5 GB, requires HF login)
#   "mistralai/Mistral-7B-v0.1"   (~14 GB, GPU recommended)

# Load the model (downloads on first run, cached afterwards)
generator = pipeline(
    "text-generation",
    model=MODEL_ID,
    device_map="auto",   # automatically uses GPU if available, else CPU
)

print(f"Model '{MODEL_ID}' loaded successfully!")

In [ ]:
# ----- Prompt the model -----
prompt = "Explain what a neural network is in simple terms:"

result = generator(
    prompt,
    max_new_tokens=200,   # how many tokens to generate
    do_sample=True,       # enable sampling (vs. greedy)
    temperature=0.7,      # creativity (0 = deterministic, 1 = max random)
    top_p=0.9,            # nucleus sampling threshold
    repetition_penalty=1.1,
)

print(result[0]["generated_text"])

In [ ]:
# ----- Chat-style prompting (for instruction-tuned models) -----
# Instruction-tuned models (names ending in -chat, -it, -Instruct) expect
# a special format. Use apply_chat_template to build it automatically.

messages = [
    {"role": "system", "content": "You are a helpful AI assistant."},
    {"role": "user",   "content": "What is the difference between GPT and BERT?"},
]

# Build the prompt string the model expects
tokenizer = generator.tokenizer
chat_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

response = generator(
    chat_prompt,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
)

# Strip the prompt prefix and print only the model's reply
reply = response[0]["generated_text"][len(chat_prompt):].strip()
print("Model reply:\n", reply)

### Tips
| Parameter | Effect |
|---|---|
| `max_new_tokens` | Controls response length |
| `temperature` | Higher → more creative / Lower → more focused |
| `top_p` | Nucleus sampling — keep tokens covering top-p probability mass |
| `repetition_penalty` | > 1.0 discourages the model from repeating itself |
| `device_map="auto"` | Uses GPU automatically; remove for CPU-only |

> **HuggingFace login** — some models (e.g. Gemma, Llama) require accepting a licence on the Hub first.  
> Run `huggingface-cli login` in your terminal and paste your access token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).